
## Core Takeaway

The code takes an array of word/token IDs, looks up their corresponding vectors in a learned dictionary, and scales those vectors up so their variance doesn't vanish during subsequent layers.

• self.emb(ids): For every integer ID passed in, PyTorch pulls out the corresponding row from the embedding matrix. If your input has a shape of (Batch_Size, Sequence_Length), the output becomes (Batch_Size, Sequence_Length, d_model).


• * math.sqrt(self.d_model): This is a specific trick from the original Transformer paper (Attention Is All You Need). As the embedding dimension (\(d_{model}\)) gets larger, the values inside the vectors naturally have smaller variances when initialized. Multiplying by \(\sqrt{d_{model}}\) scales these values up so they don't get drowned out when positional encodings are added to them later.

In [3]:
import math
import torch
import torch.nn as nn

class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.emb=nn.Embedding(vocab_size, d_model)
        self.d_model=d_model

    def forward(self, ids):
        return self.emb(ids) * math.sqrt(self.d_model)


if __name__=="__main__":
    torch.manual_seed(42)
    # SETUP MODEL HYPERPARRAMETERS:
    VOCAB_SIZE=10
    D_MODEL=4  # EACH VECTOR IS REPRESENTED BY A 4-DIM vector
    # 2. initialize model & optimizer
    model=TokenEmbedding(vocab_size=VOCAB_SIZE, d_model=D_MODEL)
    optimizer=torch.optim.SGD(model.parameters(), lr=0.1)

    # MOCK TRAINING DATA (batch of 2 sentences, each has 3 tokens)
     # Think of these integers as tokenized words: e.g., [ "I", "love", "AI" ]
    input_token=torch.tensor([
        [3, 4, 2],
         
    ])

    print("__BEFORE FORWARD PASS__")
    print("Input token IDS Shape:", input_token.shape)
    # (Batch Size, Sequence Length, D_Model)
    print("Embedding Weights (Before):\n", model.emb.weight.data)
    

__BEFORE FORWARD PASS__
Input token IDS Shape: torch.Size([2, 3])
Embedding Weights (Before):
 tensor([[ 1.9269,  1.4873,  0.9007, -2.1055],
        [ 0.6784, -1.2345, -0.0431, -1.6047],
        [-0.7521,  1.6487, -0.3925, -1.4036],
        [-0.7279, -0.5594, -0.7688,  0.7624],
        [ 1.6423, -0.1596, -0.4974,  0.4396],
        [-0.7581,  1.0783,  0.8008,  1.6806],
        [ 0.0349,  0.3211,  1.5736, -0.8455],
        [ 1.3123,  0.6872, -1.0892, -0.3553],
        [-1.4181,  0.8963,  0.0499,  2.2667],
        [ 1.1790, -0.4345, -1.3864, -1.2862]])


In [6]:
 # 4. THE FORWARD PASS (What happens inside the training loop)
model.train() # Set model to training mode
optimizer.zero_grad() # Clear any old gradients

In [8]:
# Pass integers through the model to get scaled continuous vectors
embeddings = model(input_token) 
print("\n--- AFTER FORWARD PASS ---")
print("Output Embeddings Shape:", embeddings.shape) # (Batch Size, Sequence Length, D_Model)
print("Output Vector Data:\n", embeddings) 




--- AFTER FORWARD PASS ---
Output Embeddings Shape: torch.Size([2, 3, 4])
Output Vector Data:
 tensor([[[-1.4558, -1.1189, -1.5377,  1.5249],
         [ 3.2846, -0.3192, -0.9948,  0.8792],
         [-1.5043,  3.2974, -0.7850, -2.8072]],

        [[-1.5163,  2.1566,  1.6016,  3.3612],
         [ 2.6246,  1.3743, -2.1784, -0.7106],
         [-1.4558, -1.1189, -1.5377,  1.5249]]], grad_fn=<MulBackward0>)
